# Cell QC, lane structure, and calling infection

Starts from the object written by `scripts/01_build_anndata.py`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
TABLES = Path("../results/tables")
FIGURES = Path("../results/figures")
TABLES.mkdir(parents=True, exist_ok=True)

from perturbseq.data import load_cells
from perturbseq.genesets import VIRAL_GENES
from sklearn.mixture import GaussianMixture

*from `02_qc.py`*

02 - Cell-level QC of the assigned population, reported per 10x channel.

Writes
    results/tables/02_qc_per_cell_summary.tsv     overall and per-gem-group QC
    results/tables/02_qc_by_gem_group.tsv         lane effects on depth/viral load
    results/tables/02_cells_per_element.tsv       representation per library element

What this script is for
-----------------------
Two things matter downstream and both are checked here rather than assumed:

1. Depth is not constant across lanes. Any per-element phenotype is at risk of
   confounding if elements are unevenly distributed across lanes, so this step reports
   the lane composition of every element and the depth spread between lanes.

2. The viral fraction of a cell's library is large (up to most of the library).
   That means "total UMI" is not a depth covariate: it is partly the phenotype.
   Every downstream model therefore uses host-only UMI as the offset, and this
   script quantifies how different the two denominators are.

No cells are filtered here. The authors' filtered matrix is already QC'd; this
script documents the state of the analysis set and flags anything that would
invalidate a model, rather than re-filtering.

*from `03_infection.py`*

03 - Infection-state classification, three ways, and how much of it is depth.

Writes
    results/tables/03_cell_state.tsv.gz          per-cell state, merged by later scripts
    results/tables/03_ambient_background.tsv     ambient viral rate per viral gene
    results/tables/03_classifier_agreement.tsv   concordance of the three rules
    results/tables/03_state_summary.tsv          state composition per lane
    results/tables/03_host_umi_matched_depth.tsv host shutoff at matched library size
    results/tables/03_depth_sensitivity.tsv      calls after downsampling
    results/tables/03_infected_by_depth_decile.tsv

Why this is step 03 and not step 04
-----------------------------------
On-target knockdown has to be measured in bystander cells, because infection
collapses host transcription. The infection call is therefore upstream of the
guide QC in 04, not downstream of it.

The ambient-RNA problem
-----------------------
Every droplet picks up cell-free viral RNA, so a viral count above zero is not
evidence of infection. The paper spiked in ~1% wild-type Calu-3 cells (no
dCas9, no sgRNA) to measure that background. The reference is rebuilt as

    barcodes with no guide detected, zero lentiviral reads, total UMI >= 10,000

which gives 705 cells here. That set is *not* clean: about an eighth of it sits
in the high-viral mode, i.e. real library cells whose guide was not captured.
A mean-and-SD background computed on it is therefore inflated by more than an
order of magnitude (SCV_N: mean 966, SD 2487), and a mean + 2 SD rule calls
almost nothing infected. Median-based statistics are used instead, which tolerate
that contamination, and the resulting rule is checked against the reference set.

Three classifiers
-----------------
1. `mixture`  (primary) - two-component Gaussian mixture on log10 viral
   fraction. The distribution is clearly bimodal (ambient mode near 3e-3,
   infected mode near 0.5), so the mixture finds the split without a
   hand-chosen cutoff. Posterior > 0.9 is infected, < 0.1 uninfected, the rest
   borderline.
2. `ambient`  - per viral gene, infected if the count exceeds the ambient
   expectation for that cell's library size: count_g >= max(6, k * r_g * total_umi)
   where r_g is the median ambient rate in the reference set. This is the
   depth-aware version of the paper's per-gene rule.
3. `fixed`    - viral fraction above the antimode of the bimodal distribution.
   The crude rule, included so the reader can see it barely matters.

The three are reported against each other. Downstream scripts use `mixture`,
and `07_hit_calling.py` repeats its headline test under `ambient` to show the
hit list does not depend on the choice.

The depth check
---------------
A rule with an absolute count floor calls deeply sequenced cells infected more
often, so a perturbation that changes library depth can move the apparent
infection rate with no change in infection biology. Two diagnostics: the
infected fraction across deciles of host depth, and the stability of the calls
when viral counts are binomially downsampled.

In [2]:
def q(s, p):
    return float(np.nanpercentile(s, p))

MIN_COUNT = 6              # absolute floor on viral counts for one gene

AMBIENT_K = 5.0            # fold over the depth-matched ambient expectation

REF_MIN_TOTAL_UMI = 10_000

P_HI, P_LO = 0.9, 0.1      # mixture posteriors for infected / uninfected

def fit_mixture(viral_frac: pd.Series) -> tuple[pd.Series, dict]:
    """Two-component Gaussian mixture on log10 viral fraction."""
    x = np.log10(viral_frac.clip(lower=1e-6)).to_numpy().reshape(-1, 1)
    gm = GaussianMixture(n_components=2, random_state=0, n_init=5).fit(x)
    hi = int(np.argmax(gm.means_.ravel()))
    post = gm.predict_proba(x)[:, hi]
    info = dict(
        mode_low=float(10 ** gm.means_.ravel()[1 - hi]),
        mode_high=float(10 ** gm.means_.ravel()[hi]),
        weight_high=float(gm.weights_[hi]),
    )
    # the crossover point, for the `fixed` rule
    grid = np.linspace(x.min(), x.max(), 4000).reshape(-1, 1)
    gp = gm.predict_proba(grid)[:, hi]
    info["antimode_viral_frac"] = float(10 ** grid[np.argmin(np.abs(gp - 0.5))][0])
    return pd.Series(post, index=viral_frac.index, name="p_infected"), info

a = load_cells(assigned_only=False)
obs = a.obs
asg = obs[obs.cell_set == "assigned"]
wt = obs[obs.cell_set == "wt_candidate"]

In [3]:
# ------------------------------------------------------------- overall QC
rows = []
for label, d in [("assigned", asg), ("wt_candidate", wt)]:
    rows.append(
        dict(
            population=label,
            n_cells=len(d),
            host_umi_p25=q(d.host_umi, 25),
            host_umi_median=q(d.host_umi, 50),
            host_umi_p75=q(d.host_umi, 75),
            total_umi_median=q(d.total_umi, 50),
            n_genes_median=q(d.n_genes, 50),
            pct_mito_median=q(d.pct_mito, 50),
            viral_umi_median=q(d.viral_umi, 50),
            viral_frac_median=q(d.viral_frac, 50),
            viral_frac_p90=q(d.viral_frac, 90),
            lenti_umi_median=q(d.lenti_umi, 50),
            frac_lenti_zero=float((d.lenti_umi == 0).mean()),
        )
    )
summ = pd.DataFrame(rows)
summ.to_csv(TABLES / "02_qc_per_cell_summary.tsv", sep="\t", index=False)
print(summ.to_string(index=False))

  population  n_cells  host_umi_p25  host_umi_median  host_umi_p75  total_umi_median  n_genes_median  pct_mito_median  viral_umi_median  viral_frac_median  viral_frac_p90  lenti_umi_median  frac_lenti_zero
    assigned    27882        5325.5          18066.5      28879.75           18378.0          4228.0         1.760829              65.0           0.003269        0.551087               9.0         0.145040
wt_candidate    16391         704.0           1921.0      15209.00            2438.0           958.0         0.822376              67.0           0.046275        0.207359               1.0         0.414801


In [4]:
# ------------------------------------------------- how different is total vs host
# If total UMI were a valid depth covariate it would track host UMI; it does
# not, because viral reads dominate a subset of cells.
r_all = np.corrcoef(asg.total_umi, asg.host_umi)[0, 1]
hi = asg[asg.viral_frac > 0.5]
r_hi = np.corrcoef(hi.total_umi, hi.host_umi)[0, 1] if len(hi) > 10 else np.nan
print(
    f"\ncorr(total_umi, host_umi): all assigned {r_all:.3f}; "
    f"cells with viral_frac>0.5 (n={len(hi)}) {r_hi:.3f}"
)


corr(total_umi, host_umi): all assigned 0.991; cells with viral_frac>0.5 (n=3190) 0.757


In [5]:
# ------------------------------------------------------------ lane effects
g = (
    asg.groupby("gem_group", observed=True)
    .agg(
        n_cells=("host_umi", "size"),
        host_umi_median=("host_umi", "median"),
        n_genes_median=("n_genes", "median"),
        pct_mito_median=("pct_mito", "median"),
        viral_frac_median=("viral_frac", "median"),
        viral_umi_median=("viral_umi", "median"),
        n_elements=("element", "nunique"),
    )
    .reset_index()
)
g.to_csv(TABLES / "02_qc_by_gem_group.tsv", sep="\t", index=False)
print("\nper-lane:\n", g.to_string(index=False))
spread = g.host_umi_median.max() / g.host_umi_median.min()
print(f"\nlane median host-UMI max/min = {spread:.2f}x")


per-lane:
 gem_group  n_cells  host_umi_median  n_genes_median  pct_mito_median  viral_frac_median  viral_umi_median  n_elements
        1     3725          16959.0          3981.0         1.563044           0.002410              41.0         230
        2     3673          18245.0          4232.0         1.663397           0.002643              49.0         230
        3     3346          19791.0          4406.5         1.592965           0.002325              46.0         226
        4     3558          19624.5          4393.5         1.724534           0.002431              48.0         230
        5     3264          17853.5          4285.0         1.977957           0.004625              86.0         229
        6     3557          18025.0          4240.0         1.966767           0.004339              80.0         229
        7     3243          17988.0          4304.0         1.942396           0.004121              78.0         226
        8     3516          16039.5         

In [6]:
# --------------------------------------------------- element representation
ct = (
    asg.groupby("element", observed=True)
    .agg(
        target=("target", "first"),
        is_ntc=("is_ntc", "first"),
        n_target_genes=("n_target_genes", "first"),
        n_cells=("host_umi", "size"),
        n_lanes=("gem_group", "nunique"),
        host_umi_median=("host_umi", "median"),
        viral_frac_median=("viral_frac", "median"),
    )
    .reset_index()
    .sort_values("n_cells")
)
# lane imbalance: largest share of an element's cells coming from one lane
share = (
    asg.groupby(["element", "gem_group"], observed=True).size()
    .groupby(level=0).apply(lambda s: s.max() / s.sum())
    .rename("max_lane_share")
)
ct = ct.merge(share, left_on="element", right_index=True, how="left")
ct.to_csv(TABLES / "02_cells_per_element.tsv", sep="\t", index=False)

print(
    f"\nelements: {len(ct)}  (NTC {int(ct.is_ntc.sum())}, "
    f"single-gene {int(((~ct.is_ntc) & (ct.n_target_genes == 1)).sum())}, "
    f"multi-gene {int((ct.n_target_genes > 1).sum())})"
)
print(
    f"cells per element: min {ct.n_cells.min()}, median {int(ct.n_cells.median())}, "
    f"max {ct.n_cells.max()};  {(ct.n_cells < 55).sum()} elements under 55 cells"
)
print(f"all elements present in all 8 lanes: {bool((ct.n_lanes == 8).all())}")
print(f"worst single-lane share for any element: {ct.max_lane_share.max():.2f}")

a = load_cells(assigned_only=False)
obs = a.obs
viral = [g for g in VIRAL_GENES if f"vc_{g}" in obs.columns]
vcols = [f"vc_{g}" for g in viral]


elements: 237  (NTC 22, single-gene 193, multi-gene 22)
cells per element: min 1, median 113, max 588;  47 elements under 55 cells
all elements present in all 8 lanes: False
worst single-lane share for any element: 1.00


In [7]:
# ---------------------------------------------------- ambient reference
ref = obs[
    (obs.cell_set == "wt_candidate")
    & (obs.lenti_umi == 0)
    & (obs.total_umi >= REF_MIN_TOTAL_UMI)
]
print(f"ambient reference cells: {len(ref)}")
if len(ref) < 100:
    raise SystemExit("too few reference cells to estimate the ambient background")

bg = pd.DataFrame(index=pd.Index(viral, name="viral_gene"))
bg["mean_ref"] = [ref[c].mean() for c in vcols]
bg["sd_ref"] = [ref[c].std(ddof=1) for c in vcols]
bg["median_ref"] = [ref[c].median() for c in vcols]
# median rate per UMI: robust to the infected cells contaminating the set
bg["ambient_rate"] = [float(np.median(ref[c] / ref.total_umi)) for c in vcols]
bg.to_csv(TABLES / "03_ambient_background.tsv", sep="\t")
print("\nambient background per viral gene:\n", bg.round(6).to_string())
print(
    "\nnote the mean/SD inflation: a mean + 2 SD rule would put the SCV_N "
    f"threshold at {bg.loc['SCV_N', 'mean_ref'] + 2 * bg.loc['SCV_N', 'sd_ref']:.0f} counts, "
    f"versus a median ambient rate of {bg.loc['SCV_N', 'ambient_rate']:.5f} per UMI."
)

ambient reference cells: 705

ambient background per viral gene:
               mean_ref       sd_ref  median_ref  ambient_rate
viral_gene                                                   
SCV_ORF1ab   54.601418   514.210571         4.0      0.000206
SCV_S        11.401419   167.191315         1.0      0.000036
SCV_ORF3a     8.251064    23.049988         0.0      0.000000
SCV_E         6.224114    17.512276         0.0      0.000000
SCV_M        56.445389   151.469894         2.0      0.000109
SCV_ORF6     33.567375    88.759369         1.0      0.000075
SCV_ORF7ab   67.062408   180.765289         2.0      0.000134
SCV_ORF8     38.238297   104.402206         1.0      0.000066
SCV_N       966.381531  2487.110596        45.0      0.002674

note the mean/SD inflation: a mean + 2 SD rule would put the SCV_N threshold at 5941 counts, versus a median ambient rate of 0.00267 per UMI.


In [8]:
# ------------------------------------------------------- the three rules
# 1. mixture on log10 viral fraction, fitted on the assigned cells
asg_mask = obs.cell_set == "assigned"
post_all, info = fit_mixture(obs.viral_frac.fillna(1e-6))
print(
    f"\nmixture: ambient mode at viral_frac {info['mode_low']:.4f}, "
    f"infected mode at {info['mode_high']:.3f}, "
    f"crossover at {info['antimode_viral_frac']:.4f}, "
    f"high-component weight {info['weight_high']:.3f}"
)
state_mix = np.where(post_all >= P_HI, "infected",
                     np.where(post_all <= P_LO, "uninfected", "borderline"))

# 2. depth-aware per-gene ambient rule
vc = obs[vcols].to_numpy()
lam = np.outer(obs.total_umi.to_numpy(), bg.ambient_rate.to_numpy())
thr = np.maximum(MIN_COUNT, AMBIENT_K * lam)
state_amb = np.where((vc >= thr).any(axis=1), "infected", "uninfected")

# 3. fixed cutoff at the mixture crossover
cut = info["antimode_viral_frac"]
state_fix = np.where(obs.viral_frac.fillna(0) > cut, "infected", "uninfected")

# 4. the published rule, implemented literally: per viral gene, infected if
#    the count clears both an absolute floor of 6 and mean + 2 SD of the
#    reference. Included to document why it is not used: the reference set
#    that can be rebuilt from the deposited data is contaminated by real
#    library cells whose guide was not captured, so its SD is enormous
#    (SCV_N: mean 966, SD 2487) and the threshold lands above almost every
#    cell. This is a limitation of the reconstructable reference, not a
#    claim that the published rule is wrong -- the authors had the true
#    spike-in identity, which is not recoverable from the deposited matrix.
thr_literal = np.maximum(MIN_COUNT, bg.mean_ref + 2 * bg.sd_ref).to_numpy()
state_lit = np.where((obs[vcols].to_numpy() >= thr_literal[None, :]).any(axis=1),
                     "infected", "uninfected")

obs = obs.assign(state=state_mix, p_infected=post_all.to_numpy(),
                 state_ambient=state_amb, state_fixed=state_fix,
                 state_paper_literal=state_lit)
asg = obs[asg_mask]


mixture: ambient mode at viral_frac 0.0023, infected mode at 0.100, crossover at 0.0088, high-component weight 0.460


In [9]:
# ------------------------------------------------------------ concordance
agree = pd.DataFrame(
    [
        dict(rule="mixture (primary)",
             infected_frac=float((asg.state == "infected").mean()),
             borderline_frac=float((asg.state == "borderline").mean()),
             ref_called_infected=float((obs.loc[ref.index, "state"] == "infected").mean())),
        dict(rule="ambient per-gene",
             infected_frac=float((asg.state_ambient == "infected").mean()),
             borderline_frac=0.0,
             ref_called_infected=float((obs.loc[ref.index, "state_ambient"] == "infected").mean())),
        dict(rule=f"fixed viral_frac > {cut:.3f}",
             infected_frac=float((asg.state_fixed == "infected").mean()),
             borderline_frac=0.0,
             ref_called_infected=float((obs.loc[ref.index, "state_fixed"] == "infected").mean())),
        dict(rule="published rule, literal (mean + 2 SD of reference)",
             infected_frac=float((asg.state_paper_literal == "infected").mean()),
             borderline_frac=0.0,
             ref_called_infected=float(
                 (obs.loc[ref.index, "state_paper_literal"] == "infected").mean())),
    ]
)
# pairwise agreement on the non-borderline cells
nb = asg[asg.state != "borderline"]
agree["agree_with_mixture"] = [
    1.0,
    float((nb.state_ambient == nb.state).mean()),
    float((nb.state_fixed == nb.state).mean()),
    float((nb.state_paper_literal == nb.state).mean()),
]
agree.to_csv(TABLES / "03_classifier_agreement.tsv", sep="\t", index=False)
print("\nclassifier comparison (assigned cells):\n", agree.round(4).to_string(index=False))


classifier comparison (assigned cells):
                                               rule  infected_frac  borderline_frac  ref_called_infected  agree_with_mixture
                                 mixture (primary)         0.2918           0.0967               0.1461              1.0000
                                  ambient per-gene         0.3074           0.0000               0.1603              0.9926
                          fixed viral_frac > 0.009         0.3219           0.0000               0.1560              0.9994
published rule, literal (mean + 2 SD of reference)         0.0624           0.0000               0.1163              0.7460


In [10]:
# ---------------------------------------------------------- state exports
out = asg[["element", "target", "is_ntc", "gem_group", "state", "p_infected",
           "state_ambient", "state_fixed", "state_paper_literal", "host_umi",
           "total_umi", "viral_umi", "viral_frac", "n_genes", "pct_mito"]].copy()
out.index.name = "cell_barcode"
out.to_csv(TABLES / "03_cell_state.tsv.gz", sep="\t", compression="gzip")

comp = asg.state.value_counts().rename("n_cells").to_frame()
comp["fraction"] = asg.state.value_counts(normalize=True)
print("\nassigned-cell state composition:\n", comp.round(4).to_string())

per_lane = pd.crosstab(asg.gem_group, asg.state, normalize="index").reset_index()
per_lane["n_cells"] = asg.gem_group.value_counts().reindex(per_lane.gem_group).to_numpy()
per_lane.to_csv(TABLES / "03_state_summary.tsv", sep="\t", index=False)
print("\nstate fractions per lane:\n", per_lane.round(3).to_string(index=False))
print(
    "\nLanes 1-4 and 5-8 differ systematically in viral load (see 02). The paper "
    "combined a pulsed and a non-pulsed infection condition after finding no "
    "significant difference; that split is what the lane effect reflects, which is "
    "why gem_group stays in every model in 07."
)


assigned-cell state composition:
             n_cells  fraction
state                        
uninfected    17050    0.6115
infected       8136    0.2918
borderline     2696    0.0967

state fractions per lane:
 gem_group  borderline  infected  uninfected  n_cells
        1       0.071     0.241       0.688     3725
        2       0.087     0.249       0.664     3673
        3       0.068     0.244       0.687     3346
        4       0.070     0.252       0.678     3558
        5       0.132     0.345       0.523     3264
        6       0.118     0.339       0.542     3557
        7       0.112     0.333       0.555     3243
        8       0.118     0.341       0.541     3516

Lanes 1-4 and 5-8 differ systematically in viral load (see 02). The paper combined a pulsed and a non-pulsed infection condition after finding no significant difference; that split is what the lane effect reflects, which is why gem_group stays in every model in 07.


In [11]:
# ------------------------------------------- host shutoff vs lost depth
inf = asg[asg.state == "infected"]
byst = asg[asg.state == "uninfected"]
print(
    f"\nhost UMI median: infected {inf.host_umi.median():.0f} vs bystander "
    f"{byst.host_umi.median():.0f} (ratio {inf.host_umi.median() / byst.host_umi.median():.2f})"
)
print(
    f"total UMI median: infected {inf.total_umi.median():.0f} vs bystander "
    f"{byst.total_umi.median():.0f} (ratio {inf.total_umi.median() / byst.total_umi.median():.2f})"
)
matched = (
    asg.assign(bin=pd.qcut(asg.total_umi, 10, duplicates="drop"))
    .pivot_table(index="bin", columns="state", values="host_umi",
                 aggfunc="median", observed=True)
)
matched.to_csv(TABLES / "03_host_umi_matched_depth.tsv", sep="\t")
print(
    "\nmedian host UMI within deciles of total library size "
    "(a gap here is composition, i.e. host shutoff, not sequencing depth):\n",
    matched.round(0).to_string(),
)


host UMI median: infected 2760 vs bystander 25898 (ratio 0.11)
total UMI median: infected 5658 vs bystander 25980 (ratio 0.22)

median host UMI within deciles of total library size (a gap here is composition, i.e. host shutoff, not sequencing depth):
 state               borderline  infected  uninfected
bin                                                 
(471.999, 3862.1]       3072.0    1651.0      1807.0
(3862.1, 6632.2]        5358.0    2719.0      5707.0
(6632.2, 9705.3]        8092.0    3326.0      8486.0
(9705.3, 13696.4]      11457.0    4356.0     12005.0
(13696.4, 18378.0]     15582.0    7374.0     16125.0
(18378.0, 22613.6]     19830.0   15909.0     20513.0
(22613.6, 26781.0]     24120.0   20902.0     24614.0
(26781.0, 31702.6]     28249.0   25448.0     29030.0
(31702.6, 39278.5]     33064.0   31381.0     34833.0
(39278.5, 91635.0]     43752.0   41216.0     46056.0


In [12]:
# ---------------------------------------------------- depth sensitivity
rng = np.random.default_rng(0)
rows = []
base = asg.state.to_numpy()
for frac in [1.0, 0.75, 0.5, 0.25, 0.1]:
    if frac == 1.0:
        vf = asg.viral_frac.fillna(1e-6)
    else:
        vsub = rng.binomial(asg[vcols].to_numpy().astype(int), frac).sum(axis=1)
        hsub = rng.binomial(asg.host_umi.to_numpy().astype(int), frac)
        vf = pd.Series(vsub / np.maximum(vsub + hsub, 1), index=asg.index).clip(lower=1e-6)
    st = np.where(vf > cut, "infected", "uninfected")
    rows.append(
        dict(
            downsample_frac=frac,
            infected_frac=float((st == "infected").mean()),
            agree_with_full_depth=float(
                (st[base != "borderline"] == base[base != "borderline"]).mean()
            ),
        )
    )
ds = pd.DataFrame(rows)
ds.to_csv(TABLES / "03_depth_sensitivity.tsv", sep="\t", index=False)
print("\nbinomial downsampling of the whole library:\n", ds.round(4).to_string(index=False))

dec = (
    asg.assign(decile=pd.qcut(asg.host_umi, 10, labels=False, duplicates="drop"))
    .groupby("decile", observed=True)
    .agg(
        host_umi_median=("host_umi", "median"),
        infected_frac=("state", lambda s: float((s == "infected").mean())),
        borderline_frac=("state", lambda s: float((s == "borderline").mean())),
        n=("state", "size"),
    )
    .reset_index()
)
dec.to_csv(TABLES / "03_infected_by_depth_decile.tsv", sep="\t", index=False)
print("\nby host-depth decile:\n", dec.round(3).to_string(index=False))


binomial downsampling of the whole library:
  downsample_frac  infected_frac  agree_with_full_depth
            1.00         0.3219                 0.9994
            0.75         0.3225                 0.9994
            0.50         0.3231                 0.9994
            0.25         0.3218                 0.9987
            0.10         0.3224                 0.9955



by host-depth decile:
  decile  host_umi_median  infected_frac  borderline_frac    n
      0           1419.0          0.979            0.007 2791
      1           2776.0          0.958            0.033 2787
      2           5327.0          0.619            0.279 2787
      3          10425.0          0.168            0.350 2788
      4          15578.5          0.052            0.169 2788
      5          20274.5          0.050            0.057 2788
      6          24431.0          0.031            0.037 2789
      7          28880.0          0.025            0.022 2787
      8          34703.0          0.022            0.012 2789
      9          45905.5          0.014            0.002 2788
